# Arm G order crossover — is there a scope signal to extract at all?

Every Arm G protocol through seed 110 rendered catalog order as a deterministic
function of pair-index parity: **even pairs list the in-scope path first, odd pairs
list the out-of-scope path first**. That is marginal balance across pairs, which the
validator checked, and it leaves order perfectly confounded with `inside_slot`.

Recomputation from the stored per-row data (`review_arm_g.py`) shows this is not
incidental. In the conflict condition the decision is a **perfect deterministic
function of parity** — 64/64 rows in seed 110, independently 64/64 in seed 108 — and
the layer-16 direction separates parity within the conflict condition at **AUROC
1.0000** against **0.9431** for the label.

Re-randomizing order across *different* scenarios does not fix this: order would
still vary between scenarios that differ in ids, paths and target. The identifying
design is a **within-scenario crossover** — each scenario rendered in both orders
with everything else byte-identical up to the catalog line swap, crossed with both
conditions and both A/B mappings.

The 2x2 gives three orthogonal contrasts:

| contrast | reads |
|---|---|
| condition main | the scope signal, averaged over both orders |
| order main | which path is listed first, regardless of request |
| condition x order | the requested target's catalog line |

The third is the interaction because `requested_target_line` is condition XOR order.
So the hypotheses make opposite predictions:

* **pure scope** — condition main large, interaction ~0, contrast same sign in both orders
* **pure position** — condition main ~0, interaction large, contrast **reverses sign** between orders

Note the committed data are not pure position at the margin level: the condition
contrast is +6.97 in the parity-0 cells and +2.91 in the parity-1 cells. Both are
*between-pair* comparisons, so neither identifies scope. What is fully
position-determined is the **decision**, because the threshold sits in the 2.500-unit
gap between the two clusters.

**Baseline inference only. No direction, no hook, no intervention.** This decides
whether re-extracting the direction is worth GPU time at all.

In [ ]:
# Colab supplies torch/CUDA.
print("Protocol: ARM_G_ORDER_CROSSOVER_V1")
%pip -q install "transformers==5.0.0" "accelerate==1.12.0" \
  "sentence-transformers==5.2.2" "scikit-learn==1.8.0"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
LAUNCH_DIR = "/content/drive/MyDrive/phi-map/arm-g-crossover-launch"
LAUNCH_FILES = (
    "arm_g_order_crossover.py",
    "arm_g_causal_subspace.py",
    "arm_g_causal_dose_ablation.py",
    "arm_g_causal.py",
    "arm_g_phase1.py",
    "arm_g_scenarios.py",
)
for name in LAUNCH_FILES:
    source = f"{LAUNCH_DIR}/{name}"
    assert os.path.exists(source), f"Missing {source}"
    shutil.copy2(source, f"/content/{name}")
print("Arm G crossover launch files staged: OK")

In [ ]:
ACTING_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
WORK_DIR = "/content/drive/MyDrive/phi-map/arm-g-order-crossover-seed111-v1"
PAIRS_PER_FAMILY = 16
BOOTSTRAP = 2000
BATCH_SIZE = 16

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add an HF_TOKEN secret with Llama-3.1-8B-Instruct access"

In [ ]:
from huggingface_hub import hf_hub_download
hf_hub_download(ACTING_MODEL, "config.json", token=HF_TOKEN)
print("Hugging Face model access: OK")

import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1024**3, 1), "GiB")
assert "A100" in props.name and props.total_memory >= 35 * 1024**3
assert torch.cuda.is_bf16_supported()

In [ ]:
# Deterministic tests, no GPU and no network:
#  - the eight committed protocols' prompts are byte-unchanged by the generator edit
#  - the crossed manifest differs only by the catalog line swap
#  - requested_target_line is condition XOR order
#  - the estimator recovers pure scope, pure position and the mixed case
#  - the legacy design has rank 3 where the crossed design has rank 4
import os, subprocess, sys
env = dict(os.environ)
env["HF_TOKEN"] = HF_TOKEN
base_cmd = [
    sys.executable, "/content/arm_g_order_crossover.py",
    "--model", ACTING_MODEL,
    "--output-dir", WORK_DIR,
    "--pairs-per-family", str(PAIRS_PER_FAMILY),
    "--bootstrap", str(BOOTSTRAP),
    "--batch-size", str(BATCH_SIZE),
]
subprocess.run(base_cmd + ["--self-test"], check=True, env=env)

In [ ]:
# Baseline forward pass over the order-crossed evaluation set. No intervention.
subprocess.run(base_cmd, check=True, env=env)

In [ ]:
import json
result_path = f"{WORK_DIR}/arm_g_order_crossover_result.json"
result = json.load(open(result_path))
summary = {
    "decision": result["decision"],
    "decision_reasons": result["decision_reasons"],
    "baseline": result["baseline"],
    "effects": result["effects"],
    "decision_reversals": result["decision_reversals"],
    "rates": result["rates"],
    "discrimination_by_order": result["discrimination_by_order"],
    "prior_prediction": result["prior_prediction"],
    "sample_counts": result["sample_counts"],
}
print(json.dumps(summary, indent=2))

In [ ]:
# Read the verdict against the prediction recorded before the run.
effects = result["effects"]
inside = effects["scope_at_inside_first"]
outside = effects["scope_at_outside_first"]
reversal = result["decision_reversals"]["conflict"]["decision_reversal_rate"]

print(f"condition contrast @ inside_first : {inside['mean']:+.3f}  95% CI {inside['ci_95']}")
print(f"condition contrast @ outside_first: {outside['mean']:+.3f}  95% CI {outside['ci_95']}")
print(f"condition main effect            : {effects['condition_main']['mean']:+.3f}"
      f"  95% CI {effects['condition_main']['ci_95']}")
print(f"condition x order (requested line): {effects['condition_x_order']['mean']:+.3f}"
      f"  95% CI {effects['condition_x_order']['ci_95']}")
print(f"order main effect                : {effects['order_main']['mean']:+.3f}"
      f"  95% CI {effects['order_main']['ci_95']}")
print()
print(f"conflict decisions reversed by order alone: {reversal:.1%}")
print(f"decision: {result['decision']}")
for reason in result["decision_reasons"]:
    print("  -", reason)
print()
print("Gate: re-extract the direction and run an operator x dose factorial ONLY if")
print("the condition contrast excludes zero in both orders.")

In [ ]:
import base64, gzip, json

summary_path = f"{WORK_DIR}/arm_g_order_crossover_result_summary.json"
archive_path = f"{WORK_DIR}/arm_g_order_crossover_result.json.gz.b64"
with open(summary_path, "w") as h:
    json.dump({**summary, "audits": result["audits"],
        "protocol": {"evaluation_seed": 111, "intervention": "none",
        "catalog_order_mode": "crossed", "control_label_mode": "parity_independent",
        "bootstrap_repetitions": BOOTSTRAP},
        "full_result_artifact": "arm_g_order_crossover_result.json.gz.b64"}, h, indent=1)
raw = json.dumps(result).encode("utf-8")
with open(archive_path, "wb") as h:
    h.write(base64.b64encode(gzip.compress(raw)))
print("summary:", summary_path)
print("archive:", archive_path)